In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision.transforms import v2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")



# Define transformations for the training and validation sets


transform_train = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.CenterCrop((224, 224)),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# load oxford iiit pets dataset
train_dataset = datasets.OxfordIIITPet(root = './data',split = 'trainval',transform = transform_train, download = True)
val_dataset = datasets.OxfordIIITPet(root = './data', split = 'test', transform = transform_val, download = True)

train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False)


In [40]:
model = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)


for params in model.parameters():
    params.requires_grad = False

input_features = model.fc.in_features
model.fc = nn.Linear(input_features,37)
model = model.to(device)


In [41]:
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr = 0.001)


In [42]:
training_loss = []
for epoch in range(epochs):
    model.train()
    trn_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trn_loss += loss.item()
        
    average_epoch_loss = trn_loss / len(train_loader)
    training_loss.append(average_epoch_loss)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {average_epoch_loss:.10f}")

/Users/harbakshbaath/Desktop/pets/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/10, Training Loss: 2.9089347165
Epoch 2/10, Training Loss: 1.7965564399
Epoch 3/10, Training Loss: 1.2727197573
Epoch 4/10, Training Loss: 0.9979243525
Epoch 5/10, Training Loss: 0.8259853248
Epoch 6/10, Training Loss: 0.7241549780
Epoch 7/10, Training Loss: 0.6495922681
Epoch 8/10, Training Loss: 0.5851710459
Epoch 9/10, Training Loss: 0.5618195667
Epoch 10/10, Training Loss: 0.5416034285


In [48]:
correct = 0; 
test_len = len(val_dataset)
model.eval()
with torch.no_grad():
    for batch_X,batch_y in val_loader:
        batch_correct_preds = 0
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        pred_labels = torch.argmax(y_pred,dim =1)
        batch_correct_preds += (pred_labels== batch_y).sum().item()
        correct += batch_correct_preds


print(correct)
accuracy = correct/test_len
print(f"Validation Accuracy: {accuracy*100:.10f}%")

3305
Validation Accuracy: 90.0790406105%
